# Carnet de démarrage — sujet 07 · Santander Transaction

**Objectif du sujet** : détecter la transaction d'un client.

Ce carnet est un **châssis**, pas une solution. Il fait quatre choses et s'arrête :

1. il **trouve** vos fichiers, quelle que soit la plateforme ;
2. il **charge** la table efficacement — cache Parquet construit une fois, projection de colonnes ;
3. il **profile** les données : dimensions, types, cible, manquants ;
4. il **écrit** une soumission au bon format, avec une valeur de référence volontairement nulle.

**Ce qu'il ne fait pas** : choisir votre métrique, votre découpage de validation, votre modèle, ni vos
variables. Ces décisions sont le sujet, et c'est ce que la grille note.

10,05 % de positifs. Aucune variable ne porte de signal fort.

---

## Où mettre ce carnet

| Plateforme | Ce qu'il faut faire |
|---|---|
| **Le dossier du sujet** | **https://drive.google.com/drive/folders/1oaTj6bZKs4dPJUWei0UF0WrgUp5uc5_D?usp=sharing** — téléchargez-le avant tout (le lien figure aussi dans l'énoncé) |
| **En local (VS Code, Jupyter)** | placez le carnet **dans le dossier téléchargé**, ou définissez `DONNEES_PROJET` vers ce dossier |
| **Google Colab** | déposez les fichiers dans `MyDrive/Sujet_07_Santander_Transaction/` — le carnet les trouvera ; ou téléversez-les dans `/content/` |
| **Kaggle** | créez un jeu de données avec vos fichiers, **Add Input** — `/kaggle/input` est exploré automatiquement |

**Rien n'est codé en dur** : le carnet cherche, et s'il ne trouve pas, il **dit précisément** ce qu'il a
cherché et où.


## Les fichiers de ce sujet

| Fichier | Rôle |
|---|---|
| `train.csv` | entraînement ou annexe |
| `a_decider.csv` | à prédire — la cible est absente |

**Format de sortie attendu** : `ID_code,target`.
**Colonne d'identifiant** : `ID_code`.
**Fichier à produire** : `soumission.csv`, à la racine de votre dépôt.


## Sur les données lourdes — et pourquoi pas « raylib »

`raylib` est une bibliothèque **graphique** (fenêtres, dessin, jeux) : elle ne lit aucune table. Elle
n'a rien à faire ici. Les outils qui répondent au besoin sont :

| Outil | Rôle | Présent par défaut ? |
|---|---|---|
| **Parquet + pyarrow** | format colonne compressé, lecture sélective | **oui** avec pandas récent |
| **Polars** | moteur colonne, `scan_csv` paresseux | non — `pip install polars` |
| **DuckDB** | SQL directement sur CSV/Parquet, sans chargement | non — `pip install duckdb` |

Ce carnet utilise **Parquet** (garanti), et le remplace par Polars ou DuckDB si vous les installez. Le
principe ne change pas : **ne jamais charger ce dont on n'a pas besoin**, et **payer la conversion
CSV → Parquet une seule fois**.


In [ ]:
# --- 1. Réglages --------------------------------------------------------------
from pathlib import Path
import os
import pandas as pd

# Où télécharger vos données — le dossier de votre sujet sur le Drive du cours.
LIEN_DRIVE = "__LIEN__"

# ECHANTILLON : nombre de lignes lues pour démarrer. None = tout lire.
ECHANTILLON = 200_000
DOSSIER_CACHE = Path.cwd() / "_cache_parquet"


In [ ]:
# --- 2. Trouver les données, sans rien coder en dur --------------------------
FICHIERS_ATTENDUS = ['train.csv', 'a_decider.csv']
SUJET = "Sujet_07_Santander_Transaction"


def trouver_donnees(fichiers=FICHIERS_ATTENDUS, sujet=SUJET):
    """Cherche le dossier qui contient TOUS les fichiers attendus."""
    essais = []
    pistes = []
    if os.environ.get("DONNEES_PROJET"):
        pistes.append(Path(os.environ["DONNEES_PROJET"]))
    ici = Path.cwd().resolve()
    pistes += [ici, ici / "donnees", ici / sujet]
    pistes += list(ici.parents)[:4]
    pistes += [Path("/kaggle/input"), Path("/content"), Path("/content/drive/MyDrive")]

    for racine in pistes:
        if not racine.exists():
            continue
        candidates = [racine, racine / sujet]
        if racine.name in ("input", "MyDrive"):
            candidates += [d for d in racine.iterdir() if d.is_dir()]
        for c in candidates:
            essais.append(str(c))
            if all((c / f).is_file() for f in fichiers):
                return c
    raise FileNotFoundError(
        "Dossier de données introuvable.\n"
        "Il faut un dossier contenant : " + ", ".join(fichiers) + "\n"
        "Solutions : placez le carnet à côté de ce dossier, "
        "ou définissez DONNEES_PROJET=/chemin/vers/le/dossier.\n"
        "Chemins essayés : " + str(essais[:12])
    )


DOSSIER = trouver_donnees()
print("dossier de données :", DOSSIER)
print("fichiers vus :", sorted(p.name for p in DOSSIER.iterdir() if p.is_file())[:12])


In [ ]:
# --- 3. Chargement efficace (table lourde) ------------------------------------
# Le cache Parquet s'écrit à côté du carnet, jamais dans les données : sur Kaggle,
# /kaggle/input est en lecture seule.
import pyarrow as pa
import pyarrow.csv as pvc
import pyarrow.parquet as pq


def table(nom, colonnes=None, complet=False):
    """Ouvre un CSV en profitant d'un cache Parquet.

    nom      : nom du fichier dans le dossier de données
    colonnes : liste de colonnes à garder (None = toutes) — le levier principal
    morceaux : si un entier, rend un itérateur de DataFrames de cette taille
    """
    source = DOSSIER / nom
    if not source.is_file():
        raise FileNotFoundError(f"{nom} introuvable dans {DOSSIER}")

    # lecture d'un échantillon : pas de cache, on veut seulement voir
    if ECHANTILLON and not complet:
        lots, reste = [], ECHANTILLON
        for lot in pvc.open_csv(str(source), read_options=pvc.ReadOptions(block_size=1 << 25)):
            lots.append(lot)
            reste -= lot.num_rows
            if reste <= 0:
                break
        t = pa.Table.from_batches(lots)
        if colonnes:
            t = t.select([c for c in colonnes if c in t.column_names])
        return t.to_pandas()

    # cache Parquet, construit une fois
    cache = DOSSIER_CACHE / (Path(nom).stem + ".parquet")
    if not cache.is_file():
        DOSSIER_CACHE.mkdir(parents=True, exist_ok=True)
        print(f"construction du cache Parquet : {cache.name} (une seule fois)")
        ecrivain = None
        try:
            for lot in pvc.open_csv(str(source), read_options=pvc.ReadOptions(block_size=1 << 26)):
                if colonnes:
                    lot = lot.select([c for c in colonnes if c in lot.schema.names])
                if ecrivain is None:
                    ecrivain = pq.ParquetWriter(cache, lot.schema)
                ecrivain.write_batch(lot)
        finally:
            if ecrivain is not None:
                ecrivain.close()
    fichier = pq.ParquetFile(cache)
    return fichier.read(columns=colonnes).to_pandas()


def table_par_morceaux(nom, colonnes=None, morceaux=200_000):
    """Itère sur la table par paquets — pour les traitements qui ne tiennent pas en mémoire."""
    table(nom, colonnes=colonnes)          # garantit que le cache existe
    cache = DOSSIER_CACHE / (Path(nom).stem + ".parquet")
    for lot in pq.ParquetFile(cache).iter_batches(batch_size=morceaux):
        yield lot.to_pandas()


In [ ]:
# --- 4. Charger et regarder ---------------------------------------------------
entrainement = table("train.csv")
print("dimensions :", entrainement.shape)
print("colonnes :", len(entrainement.columns))
entrainement.head()


In [ ]:
# --- 5. Profil de la cible ----------------------------------------------------
CIBLE = "target"
if CIBLE in entrainement.columns:
    serie = entrainement[CIBLE]
    print("type :", serie.dtype)
    print("manquants :", int(serie.isna().sum()))
    print(serie.describe())
    if serie.nunique() <= 20:
        print(serie.value_counts(normalize=True).sort_index())
else:
    print(f"la cible {CIBLE} est absente de cette table — normal pour la table à prédire")


In [ ]:
# --- 6. Soumission de RÉFÉRENCE (format seulement) ----------------------------
# Cette valeur n'a AUCUNE prétention prédictive : elle vérifie le format.
COLONNES_SORTIE = ['ID_code', 'target']


def ecrire_soumission(identifiants, valeurs=None, chemin="soumission.csv"):
    """Écrit le fichier attendu. `valeurs` peut être une colonne, ou une matrice de probabilités."""
    sortie = pd.DataFrame({"ID_code": identifiants})
    if len(COLONNES_SORTIE) == 2:
        sortie["target"] = valeurs
    else:
        n = len(COLONNES_SORTIE) - 1
        matrice = valeurs if valeurs is not None else [[1.0 / n] * n for _ in identifiants]
        for j, colonne in enumerate(COLONNES_SORTIE[1:]):
            sortie[colonne] = [ligne[j] for ligne in matrice]
    sortie.to_csv(chemin, index=False)
    print(f"écrit : {chemin} · {len(sortie):,} lignes · colonnes {list(sortie.columns)}")
    if len(COLONNES_SORTIE) > 2:
        print("somme par ligne (doit valoir 1) :", sortie.iloc[:, 1:].sum(axis=1).round(6).unique()[:3])
    return sortie


a_predire = table("a_decider.csv", colonnes=["ID_code"], complet=True)
identifiants = a_predire["ID_code"].drop_duplicates().to_numpy()
print("identifiants à prédire :", len(identifiants))
ecrire_soumission(identifiants)


In [ ]:
# --- 7. Le travail commence ICI -----------------------------------------------
# À vous de décider, et de justifier :
#   - la métrique, et ce qu'elle coûte ;
#   - le découpage de validation ;
#   - les variables, leur encodage, leur imputation ;
#   - le modèle, son réglage, son seuil.
# Ce carnet ne le fera pas à votre place : c'est l'objet du projet.
raise NotImplementedError(
    "Carnet de démarrage : la modélisation est à votre charge. Écrivez votre pipeline à partir d'ici."
)
